In [0]:
%python

from pyspark.sql import functions as F
import json

base = "/Volumes/blueprint_spark/bronze/bronze/streaming"
entrada = f"{base}/entrada"            # pasta onde os arquivos "chegam"
checkpoints = f"{base}/checkpoints"    # onde o stream guarda o progresso

tabela_bronze = "blueprint_spark.bronze.pedidos_stream"
tabela_silver = "blueprint_spark.bronze.pedidos_stream_silver"

schema = "pedido_id INT, cidade STRING, valor DOUBLE, data STRING"

dbutils.fs.mkdirs(entrada)

# Simula a chegada de um arquivo novo com 5 pedidos
def gerar_lote(n):
    cidades = ["Salvador", "Recife", "Curitiba"]
    linhas = [
        json.dumps({"pedido_id": n * 100 + i, "cidade": cidades[i % 3], "valor": 50.0 + i * 40, "data": "2026-11-01"})
        for i in range(5)
    ]
    dbutils.fs.put(f"{entrada}/lote_{n}.json", "\n".join(linhas), True)

In [0]:
gerar_lote(1)

Wrote 388 bytes.


In [0]:
df_batch = spark.read.json(entrada,schema=schema)
display(df_batch)

pedido_id,cidade,valor,data
100,Salvador,50.0,2026-11-01
101,Recife,90.0,2026-11-01
102,Curitiba,130.0,2026-11-01
103,Salvador,170.0,2026-11-01
104,Recife,210.0,2026-11-01


In [0]:
df_stream = spark.readStream.json(entrada,schema=schema)
print(df_stream.isStreaming)

True


In [0]:
%python

gerar_lote(1)

# LEITURA: a pasta vira uma fonte contínua (schema é obrigatório)
df_stream = spark.readStream.json(entrada, schema=schema)

# ESCRITA: todo stream precisa de um destino
df_stream.writeStream \
    .option("checkpointLocation", f"{checkpoints}/aula02") \
    .trigger(availableNow=True) \
    .toTable("blueprint_spark.bronze.aula02") \
    .awaitTermination()

spark.table("blueprint_spark.bronze.aula02").show()
# 5 linhas: o arquivo que estava na pasta

Wrote 388 bytes.
+---------+--------+-----+----------+
|pedido_id|  cidade|valor|      data|
+---------+--------+-----+----------+
|      100|Salvador| 50.0|2026-11-01|
|      101|  Recife| 90.0|2026-11-01|
|      102|Curitiba|130.0|2026-11-01|
|      103|Salvador|170.0|2026-11-01|
|      104|  Recife|210.0|2026-11-01|
+---------+--------+-----+----------+



In [0]:
%python

# Chegou um arquivo novo
gerar_lote(2)

# Roda o mesmo stream de novo
df_stream.writeStream \
    .option("checkpointLocation", f"{checkpoints}/aula02") \
    .trigger(availableNow=True) \
    .toTable("blueprint_spark.bronze.aula02") \
    .awaitTermination()

spark.table("blueprint_spark.bronze.aula02").show()
# 10 linhas: o stream leu SÓ o arquivo novo

Wrote 388 bytes.
+---------+--------+-----+----------+
|pedido_id|  cidade|valor|      data|
+---------+--------+-----+----------+
|      200|Salvador| 50.0|2026-11-01|
|      201|  Recife| 90.0|2026-11-01|
|      202|Curitiba|130.0|2026-11-01|
|      203|Salvador|170.0|2026-11-01|
|      204|  Recife|210.0|2026-11-01|
|      100|Salvador| 50.0|2026-11-01|
|      101|  Recife| 90.0|2026-11-01|
|      102|Curitiba|130.0|2026-11-01|
|      103|Salvador|170.0|2026-11-01|
|      104|  Recife|210.0|2026-11-01|
+---------+--------+-----+----------+



In [0]:
%python

# Stream com agregação: total vendido por cidade
df_total = (
    spark.readStream.json(entrada, schema=schema)
    .groupBy("cidade")
    .agg(F.round(F.sum("valor"), 2).alias("total"))
)

# complete: a cada execução, a tabela inteira é reescrita com o resultado atualizado
df_total.writeStream \
    .outputMode("complete") \
    .option("checkpointLocation", f"{checkpoints}/aula03") \
    .trigger(availableNow=True) \
    .toTable("blueprint_spark.bronze.aula03_total") \
    .awaitTermination()

spark.table("blueprint_spark.bronze.aula03_total").show()
# 3 linhas: uma por cidade, somando os lotes 1 e 2

+--------+-----+
|  cidade|total|
+--------+-----+
|Curitiba|260.0|
|Salvador|440.0|
|  Recife|600.0|
+--------+-----+



In [0]:
%python

# Chegou mais um arquivo
gerar_lote(3)

df_total.writeStream \
    .outputMode("complete") \
    .option("checkpointLocation", f"{checkpoints}/aula03") \
    .trigger(availableNow=True) \
    .toTable("blueprint_spark.bronze.aula03_total") \
    .awaitTermination()

spark.table("blueprint_spark.bronze.aula03_total").show()
# Continuam 3 linhas, mas os totais AUMENTARAM: o resultado foi recalculado

Wrote 388 bytes.
+--------+-----+
|  cidade|total|
+--------+-----+
|Curitiba|390.0|
|Salvador|660.0|
|  Recife|900.0|
+--------+-----+



In [0]:
%python

# Cenário do zero: 2 arquivos na pasta
dbutils.fs.rm(base, recurse=True)
dbutils.fs.mkdirs(entrada)
spark.sql("DROP TABLE IF EXISTS blueprint_spark.bronze.aula04_batch")
spark.sql("DROP TABLE IF EXISTS blueprint_spark.bronze.aula04_stream")
gerar_lote(1)
gerar_lote(2)

# Mesma carga nos dois jeitos. Cada linha guarda o arquivo de origem.
def carga_batch():
    spark.read.json(entrada, schema=schema) \
        .withColumn("arquivo", F.col("_metadata.file_name")) \
        .write.mode("append").saveAsTable("blueprint_spark.bronze.aula04_batch")

def carga_stream():
    spark.readStream.json(entrada, schema=schema) \
        .withColumn("arquivo", F.col("_metadata.file_name")) \
        .writeStream \
        .option("checkpointLocation", f"{checkpoints}/aula04") \
        .trigger(availableNow=True) \
        .toTable("blueprint_spark.bronze.aula04_stream") \
        .awaitTermination()

def comparar():
    print("BATCH")
    spark.table("blueprint_spark.bronze.aula04_batch").groupBy("arquivo").count().orderBy("arquivo").show()
    print("STREAM")
    spark.table("blueprint_spark.bronze.aula04_stream").groupBy("arquivo").count().orderBy("arquivo").show()

# 1ª execução: os dois carregam os 2 arquivos
carga_batch()
carga_stream()
comparar()
# BATCH e STREAM iguais: lote_1 = 5, lote_2 = 5

Wrote 388 bytes.
Wrote 388 bytes.
BATCH
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|    5|
|lote_2.json|    5|
+-----------+-----+

STREAM
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|    5|
|lote_2.json|    5|
+-----------+-----+



In [0]:
%python

# Chegou um arquivo novo. Rodamos as duas cargas de novo.
gerar_lote(3)
carga_batch()
carga_stream()
comparar()
# BATCH:  lote_1 = 10, lote_2 = 10, lote_3 = 5  -> DUPLICOU, porque releu a pasta inteira
# STREAM: lote_1 = 5,  lote_2 = 5,  lote_3 = 5  -> leu SÓ o arquivo novo

Wrote 388 bytes.
BATCH
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|   10|
|lote_2.json|   10|
|lote_3.json|    5|
+-----------+-----+

STREAM
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|    5|
|lote_2.json|    5|
|lote_3.json|    5|
+-----------+-----+



In [0]:
%python

# Por que o stream não duplicou? O checkpoint anotou o que já foi lido.
display(dbutils.fs.ls(f"{checkpoints}/aula04"))           # offsets/, commits/, sources/

print(dbutils.fs.head(f"{checkpoints}/aula04/sources/0/0"))   # execução 1: lote_1 e lote_2
print(dbutils.fs.head(f"{checkpoints}/aula04/sources/0/1"))   # execução 2: só lote_3

path,name,size,modificationTime
dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/checkpoints/aula04/commits/,commits/,0,1793577544295
dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/checkpoints/aula04/metadata,metadata,45,1793577470000
dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/checkpoints/aula04/offsets/,offsets/,0,1793577544295
dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/checkpoints/aula04/sources/,sources/,0,1793577544295


v1
{"path":"dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/entrada/lote_1.json","timestamp":1793577463000,"batchId":0}
{"path":"dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/entrada/lote_2.json","timestamp":1793577464000,"batchId":0}
v1
{"path":"dbfs:/Volumes/blueprint_spark/bronze/bronze/streaming/entrada/lote_3.json","timestamp":1793577490000,"batchId":1}


In [0]:
%python

# Prova final: apagando o checkpoint, o stream "esquece" tudo
dbutils.fs.rm(f"{checkpoints}/aula04", recurse=True)
carga_stream()
comparar()
# STREAM agora também duplicou: releu os 3 arquivos do zero

BATCH
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|   10|
|lote_2.json|   10|
|lote_3.json|    5|
+-----------+-----+

STREAM
+-----------+-----+
|    arquivo|count|
+-----------+-----+
|lote_1.json|   10|
|lote_2.json|   10|
|lote_3.json|   10|
+-----------+-----+



In [0]:
%python

# Cenário do zero
dbutils.fs.rm(base, recurse=True)
dbutils.fs.mkdirs(entrada)
for t in ["p_bronze", "p_silver", "p_gold"]:
    spark.sql(f"DROP TABLE IF EXISTS blueprint_spark.bronze.{t}")

bronze = "blueprint_spark.bronze.p_bronze"
silver = "blueprint_spark.bronze.p_silver"
gold   = "blueprint_spark.bronze.p_gold"

def rodar_pipeline():
    # BRONZE: arquivos -> tabela, dado bruto
    spark.readStream.json(entrada, schema=schema) \
        .withColumn("arquivo", F.col("_metadata.file_name")) \
        .writeStream \
        .option("checkpointLocation", f"{checkpoints}/bronze") \
        .trigger(availableNow=True) \
        .toTable(bronze).awaitTermination()

    # SILVER: lê a bronze como stream, limpa e marca a hora do processamento
    spark.readStream.table(bronze) \
        .withColumn("cidade", F.upper("cidade")) \
        .withColumn("data", F.to_date("data")) \
        .withColumn("processado_em", F.current_timestamp()) \
        .writeStream \
        .option("checkpointLocation", f"{checkpoints}/silver") \
        .trigger(availableNow=True) \
        .toTable(silver).awaitTermination()

    # GOLD: lê a silver como stream e mantém o total por cidade sempre atualizado
    spark.readStream.table(silver) \
        .groupBy("cidade") \
        .agg(F.count("*").alias("pedidos"), F.sum("valor").alias("total")) \
        .writeStream \
        .outputMode("complete") \
        .option("checkpointLocation", f"{checkpoints}/gold") \
        .trigger(availableNow=True) \
        .toTable(gold).awaitTermination()

In [0]:
%python

# Simulando o dia: um arquivo chega, o pipeline roda. Três vezes.
for n in [1, 2, 3]:
    gerar_lote(n)
    rodar_pipeline()
    print(f"Rodada {n}: bronze = {spark.table(bronze).count()} | silver = {spark.table(silver).count()}")
    spark.table(gold).orderBy("cidade").show()
# Bronze e silver crescem 5 por rodada. A gold é recalculada e os totais sobem.

Wrote 388 bytes.
Rodada 1: bronze = 5 | silver = 5
+--------+-------+-----+
|  cidade|pedidos|total|
+--------+-------+-----+
|CURITIBA|      1|130.0|
|  RECIFE|      2|300.0|
|SALVADOR|      2|220.0|
+--------+-------+-----+

Wrote 388 bytes.
Rodada 2: bronze = 10 | silver = 10
+--------+-------+-----+
|  cidade|pedidos|total|
+--------+-------+-----+
|CURITIBA|      2|260.0|
|  RECIFE|      4|600.0|
|SALVADOR|      4|440.0|
+--------+-------+-----+

Wrote 388 bytes.
Rodada 3: bronze = 15 | silver = 15
+--------+-------+-----+
|  cidade|pedidos|total|
+--------+-------+-----+
|CURITIBA|      3|390.0|
|  RECIFE|      6|900.0|
|SALVADOR|      6|660.0|
+--------+-------+-----+



In [0]:
%python

# A prova de que é streaming: quantas linhas cada execução processou na silver
spark.table(silver).groupBy("processado_em", "arquivo").count().orderBy("processado_em").show(truncate=False)
# 3 horários diferentes, 5 linhas cada, um arquivo por horário.
# Em batch, a 3ª rodada teria reprocessado as 15 linhas. Aqui processou só as 5 novas.

+-----------------------+-----------+-----+
|processado_em          |arquivo    |count|
+-----------------------+-----------+-----+
|2026-11-02 00:04:34.526|lote_1.json|5    |
|2026-11-02 00:05:00.251|lote_2.json|5    |
|2026-11-02 00:05:26.63 |lote_3.json|5    |
+-----------------------+-----------+-----+

